In [1]:
import nest_asyncio
nest_asyncio.apply()          # lets agent.run_sync() work inside Jupyter's running event loop

# 04 · The analyst agent: Pydantic AI + `run_python` + inline skills

One agent, GPT-5.6 Luna on Bedrock (VAYU's model and IAM path), five tools, six skills written as plain text in this notebook. A conversation is a
Python list of messages; `ask()` appends to it, so follow-up questions see everything that came before.
Nothing here is loaded from outside the notebook except `sandbox/sandbox.py` (notebook 02) and the parquet (notebook 01).

**Needs:** the `aeronet-sandbox` image (notebook 02), this EC2 box's instance role (the one VAYU uses), and

```
pip install "pydantic-ai-slim[bedrock,bedrock-mantle]"
```

1. Tools — the three functions from notebook 03
2. Skills — six analysis procedures the model reads on every call
3. System prompt — rules + data dictionary + sandbox environment + skills
4. Model + agent
5. `ask()` — one conversation, many questions
6. Try it
7. What the model actually saw
8. Save / resume a conversation
9. Cleanup

In [2]:
import json, os, re, sys
os.environ["PYDANTIC_AI_NO_BANNER"] = "1"   # no Logfire advert on first use
from pathlib import Path
from IPython.display import Image, display
import pandas as pd
import pyarrow.parquet as pq

sys.path.insert(0, "../sandbox")
import sandbox as sb

CONVERSATION_ID = "conv_004"          # container name + folder conversation_data/conv_004/

## 1 · Tools

Five tools. The first three are the sandbox functions from notebook 03. `list_sites` and `describe_columns` are
deterministic lookups computed on the host from the parquet: the two things the system prompt cannot know (which sites
exist, with what record, and which wavelengths a site actually measured), so the model does not spend `run_python`
calls finding out. The docstring is the tool description the model reads; the type hints become the argument schema
(Pydantic AI validates arguments and asks the model to retry when they are wrong).

In [3]:
def run_python(code: str, timeout: int = 60) -> str:
    '''Run Python code in this conversation's sandbox and return what happened.
    Pre-imported: numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns.
    DATA = path of the AERONET AOD Level 2 daily parquet (read-only; read a column subset, the full table is ~1 GB).
    Variables persist between calls. Open matplotlib figures are saved as PNG automatically.
    Write any export (parquet, csv, png) under /workspace/. No network. 2 GB RAM, 2 CPUs.
    The value of the last expression is returned as a preview, like in a notebook.'''
    if not sb.running(CONVERSATION_ID):
        sb.start(CONVERSATION_ID)
    r = sb.exec_code(CONVERSATION_ID, code, timeout)
    out = []
    if r.get("stdout"):       out.append(r["stdout"].rstrip())
    if r.get("stderr"):       out.append("stderr:\n" + r["stderr"].rstrip())
    if r.get("error"):        out.append("ERROR: " + r["error"])
    if r.get("result"):       out.append(r["result"])
    if r.get("figures"):      out.append("figures saved: " + ", ".join(r["figures"]))
    if r.get("new_files"):    out.append("files written: " + ", ".join(r["new_files"]))
    if r.get("vars_new"):     out.append(f"new variables: {r['vars_new']}")
    if r.get("vars_changed"): out.append(f"changed variables: {r['vars_changed']}")
    if "elapsed_s" in r:      out.append(f"[{r['elapsed_s']} s, {r['rss_mb']} MB RSS]")
    return "\n".join(out) or "(no output)"


def kernel_state() -> str:
    '''Variables (type and shape), files in /workspace and memory use of this conversation's sandbox.'''
    return json.dumps(sb.state(CONVERSATION_ID), indent=1)


def reset_kernel() -> str:
    '''Delete all variables and open figures. Files in /workspace are kept.'''
    sb.reset(CONVERSATION_ID)
    return "kernel reset: no variables"


# ── two deterministic tools: computed on the host from the parquet, no LLM, no sandbox ──────────────
_SITES = None

def _sites():
    '''One row per site: location, record span, days with data. Read once (~10 s), then cached.'''
    global _SITES
    if _SITES is None:
        cols = ["AERONET_Site_Name", "Site_Latitude_Degrees", "Site_Longitude_Degrees", "Site_Elevation_m", "date"]
        df = pd.read_parquet(sb.PARQUET, columns=cols)
        g = df.groupby("AERONET_Site_Name")
        _SITES = pd.DataFrame({"lat": g["Site_Latitude_Degrees"].first().round(3),
                               "lon": g["Site_Longitude_Degrees"].first().round(3),
                               "elev_m": g["Site_Elevation_m"].first().round(0).astype(int),
                               "first": g["date"].min().dt.strftime("%Y-%m-%d"),
                               "last": g["date"].max().dt.strftime("%Y-%m-%d"),
                               "days": g.size()}).sort_values("days", ascending=False)
    return _SITES


def list_sites(name_contains: str = "", min_days: int = 0, limit: int = 40) -> str:
    '''Find AERONET sites. Returns name, latitude, longitude, elevation (m), first and last day with data, and the
    number of days with Level 2.0 data, sorted by record length. name_contains is case-insensitive
    (e.g. "kanpur", "GSFC", "_Island"). Use this before any site analysis to get the exact site name.'''
    s = _sites()
    if name_contains:
        s = s[s.index.str.contains(name_contains, case=False, regex=False)]
    s = s[s["days"] >= min_days]
    if s.empty:
        return f"no site matches name_contains={name_contains!r}, min_days={min_days}; try a shorter fragment"
    head = f"{len(s)} site(s) match" + (f", showing the {limit} with the longest records" if len(s) > limit else "") + "\n"
    return head + s.head(limit).to_string()


def _describe(col):
    if m := re.fullmatch(r"AOD_(\d+)nm", col):                       return f"aerosol optical depth at {m[1]} nm (unitless)"
    if m := re.fullmatch(r"Angstrom_Exponent_(\d+)_(\d+)nm(_Polar)?", col):
        return f"Ångström exponent {m[1]}–{m[2]} nm" + (" (polar instrument)" if m[3] else "")
    if m := re.fullmatch(r"N_(.+)", col):                             return f"number of measurements in the daily {m[1]}"
    return {"AERONET_Site_Name": "site name", "datetime_utc": "day stamp, 12:00 UTC", "date": "day",
            "year": "year", "month": "month", "Day_of_Year": "day of year 1–366",
            "Site_Latitude_Degrees": "latitude (deg)", "Site_Longitude_Degrees": "longitude (deg)",
            "Site_Elevation_m": "elevation (m)", "Precipitable_Water_cm": "column water vapour (cm)",
            "Data_Quality_Level": "always lev20", "AERONET_Instrument_Number": "sun-photometer id",
            "PI": "site principal investigator", "PI_Email": "PI e-mail"}.get(col, "")


def describe_columns(site: str = "") -> str:
    '''Every column of the AERONET table with its meaning, dtype and how many days have a value.
    Without a site: the whole table (all sites). With a site (exact name from list_sites): that site only —
    use it to see which wavelengths the site measured before choosing one.'''
    pf = pq.ParquetFile(sb.PARQUET)
    if site:
        t = pq.read_table(sb.PARQUET, filters=[("AERONET_Site_Name", "==", site)])
        if t.num_rows == 0:
            return f"no site named {site!r}; use list_sites to find the exact name"
        n_days, nulls = t.num_rows, {c: t.column(c).null_count for c in t.column_names}
    else:
        n_days = pf.metadata.num_rows
        nulls = {c: 0 for c in pf.schema_arrow.names}
        for rg in range(pf.metadata.num_row_groups):
            for i in range(pf.metadata.num_columns):
                col = pf.metadata.row_group(rg).column(i)
                nulls[col.path_in_schema] += col.statistics.null_count if col.statistics else 0
    where = f"site {site}" if site else "all sites"
    lines = [f"{where}: {n_days} rows (site-days)", f"{'column':34} {'dtype':13} {'days_with_value':>15} {'cover':>6}  meaning"]
    for c in pf.schema_arrow.names:
        have = n_days - nulls[c]
        lines.append(f"{c:34} {str(pf.schema_arrow.field(c).type):13} {have:>15} {100 * have / n_days:>5.1f}%  {_describe(c)}")
    return "\n".join(lines)


TOOLS = [run_python, kernel_state, reset_kernel, list_sites, describe_columns]

## 2 · Skills

A skill is a short written procedure: when to use it, the steps, the thresholds, the caveats. They go into the system
prompt as text, so the model reads all of them on every call (that part of the prompt is cached, see section 4).
Edit them here; nothing else has to change. The sandbox has `numpy`, `pandas`, `scipy`, `statsmodels`, `matplotlib`
and `seaborn`, so the statistics name the library function to call.

In [4]:
SKILLS = {}

SKILLS["data_access"] = """
WHEN: the first step of every analysis, before any statistic.
1. list_sites(name_contains=...) gives the exact site name (case-sensitive, underscores: "Mauna_Loa"), coordinates
   and record span; describe_columns(site=...) shows which wavelengths that site has and on how many days.
   Then load only the columns you need, filtered by site, e.g.
   pd.read_parquet(DATA, columns=["AERONET_Site_Name","date","year","month","AOD_500nm","N_AOD_500nm"],
                   filters=[("AERONET_Site_Name","==","GSFC")])
2. Describe the record before using it: days per year, gaps, and the coverage of the wavelength you chose (every
   column is NaN where that wavelength was not measured or failed Level 2.0 screening).
   Coverage is uneven: instruments are moved, break, and are recalibrated; gaps are normal, not errors.
3. One row = one day (daily mean of all Level 2.0 measurements of that day, stamped 12:00 UTC). N_<col> is how
   many measurements went into that mean; a day with N=1 is a real but weak estimate — say so if it matters.
4. Level 2.0 = cloud-screened AND quality-assured with final calibration (Version 3, Giles et al. 2019). You do not
   need extra cloud screening. Do not "clean" outliers: high AOD days are real events (dust, smoke).
5. Wavelengths: 500 nm is the standard reporting wavelength; 440/675/870/1020 nm are on every instrument;
   340/380 nm and 1640 nm only on some. AOD_531nm and AOD_551nm etc. are rarely populated. Check before use.
6. Keep memory small: never load all 73 columns for all sites at once (~1 GB); aggregate per site, then combine.
"""

SKILLS["climatology"] = """
WHEN: "typical", "average", "seasonal cycle", "monthly/annual mean", "how much AOD at site X".
1. Aggregate daily -> monthly means only when the month has enough days; default rule: >= 5 daily values per
   calendar month, and an annual mean needs >= 9 valid months. State the rule you used and how many months/years
   it removed.
2. AOD is right-skewed (roughly log-normal): report mean AND median (and the 10th/90th percentiles when
   describing variability). Use the mean for comparison with satellite/model climatologies, the median for
   "typical day". Ångström exponent is roughly symmetric; the mean is fine.
3. Seasonal cycle = group by calendar month across all years (multi-year monthly climatology); show N years per
   month. Standard deviation across years = interannual variability; std of daily values within the month = day-to-day
   variability. Say which one you plotted.
4. Multi-year climatologies over a changing record are biased if some years are missing whole seasons: check that
   each month-of-year is represented by a similar number of years, otherwise say so.
5. Plot: line or bar of the 12 monthly values with error bars (std or IQR), y-axis label "AOD 500 nm", title with
   site name and period, N per month annotated or in a table. Save under /workspace.
"""

SKILLS["trend"] = """
WHEN: "trend", "is AOD increasing/decreasing", "change over years", "long-term".
1. Requirements: >= 8 years of data with >= 9 valid months each; otherwise say the record is too short and stop at
   a description. Check for large gaps or an instrument change (AERONET_Instrument_Number) that splits the record.
2. Remove the seasonal cycle first: monthly means (>= 5 days) -> monthly anomalies = value minus the multi-year
   mean of that calendar month. Fit the trend to the anomaly time series with time in decimal years, NaN months
   dropped.
3. Use the Theil–Sen slope with the Mann–Kendall test (robust to the skewed distribution and outliers):
     from scipy import stats
     slope, intercept, lo, hi = stats.theilslopes(y, t, alpha=0.95)     # slope + 95 % confidence interval
     tau, p = stats.kendalltau(t, y)                                     # Mann–Kendall on a time series
   Cross-check with OLS that allows for autocorrelated residuals:
     import statsmodels.api as sm
     ols = sm.OLS(y, sm.add_constant(t)).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
   Report the OLS slope and its HAC p-value next to the Theil–Sen result; if they disagree on significance, say so.
4. Report: slope in AOD per year AND in % per year relative to the record mean, the confidence interval, n months
   used, p-value, the period. "Significant" = p < 0.05, and say so explicitly either way.
5. Caveats to state: monthly anomalies are autocorrelated (the Kendall p is optimistic, the HAC p is the honest
   one); a trend can be driven by the first or last years — plot the anomalies with the fitted line and look;
   attribution (emissions, dust, fires) is outside the data — do not claim causes.
"""

SKILLS["aerosol_type"] = """
WHEN: "what kind of aerosol", "dust or smoke", "fine vs coarse", "Ångström exponent", "AOD at 550 nm".
1. Use the Angstrom_Exponent_440_870nm column (computed per measurement from the spectral AOD, then averaged);
   do not recompute it from daily-mean AODs unless it is missing.
2. Interpretation (rule of thumb, Eck et al. 1999; Dubovik et al. 2002): AE < 0.75 -> coarse-mode dominated
   (desert dust, sea salt); AE > 1.5 -> fine-mode dominated (smoke, urban/industrial pollution); 0.75–1.5 -> mixed.
   Combined with AOD: high AOD (>= 0.4 at 500 nm) & low AE = dust event; high AOD & high AE = biomass burning or
   pollution; low AOD (< 0.1) & low AE = clean marine or background.
3. AE is noisy when AOD is low: exclude days with AOD_440nm < 0.15 before classifying, and say how many days that
   removed.
4. Show the AOD–AE relationship as a scatter (x = AE 440–870, y = AOD 500 nm, log y), colored by month or season,
   plus a table of the fraction of days in each class. This is a classification of daily means, not of single plumes.
5. AOD at a wavelength that is not measured (e.g. 550 nm): Ångström power law
   AOD(l) = AOD(500) * (l/500) ** (-AE) with AE = Angstrom_Exponent_440_870nm; valid only for 440–870 nm.
   Say that the value is interpolated.
"""

SKILLS["episodes"] = """
WHEN: "event", "episode", "extreme days", "dust storm", "smoke", "when was AOD highest".
1. "High" must be relative to the site's own climatology: flag a day when AOD 500 nm exceeds the site's
   multi-year 95th percentile for that calendar month (default), or a fixed value the user gives. State the
   threshold.
2. Group consecutive flagged days into episodes; report start, end, duration, peak AOD and its date, mean AE
   during the episode (for the likely aerosol type, see the aerosol_type skill), and N days.
3. Rank episodes by peak or by cumulative AOD above the threshold; list the top 10 as a table.
4. Show the daily series for the year(s) of interest with the threshold line and the episodes shaded.
5. A missing day inside an episode is likely cloud (no Level 2.0 data), not a gap in the event; say so rather than
   splitting the episode, unless the gap is > 2 days.
6. Do not name the source (a specific fire, storm) — the data cannot show it; describe timing, magnitude and type.
"""

SKILLS["site_comparison"] = """
WHEN: two or more sites, "compare", "regional", "which site is dustier", "difference between".
1. Compare only over the overlapping period and the same wavelength; state the common period and N days per site.
   If overlap < 2 years, compare climatologies instead and say they are from different years.
2. Report per site: coordinates, elevation, period, N, mean, median, 90th percentile of AOD 500 nm and mean AE
   440–870; then the difference of means/medians. Elevation matters: a mountain site (Mauna_Loa, Izana, > 2 km)
   sees only the free troposphere and is not comparable with a surface site nearby.
3. For paired daily differences use only days both sites have data (inner join on date).
4. Plots: same y-axis for all sites; monthly climatology lines on one axis, or box plots per site; a map (lon/lat
   scatter with site names) helps when there are more than 3 sites.
5. Distances between sites (haversine) belong in the answer when the user asks about "nearby".
"""

SKILLS_TEXT = "\n".join(f"### skill: {name}\n{text.strip()}\n" for name, text in SKILLS.items())
print(f"{len(SKILLS)} skills, {len(SKILLS_TEXT.split())} words")

6 skills, 1273 words


## 3 · System prompt

Four blocks, always in this order: rules, data dictionary, sandbox environment (read from the running container so it
never drifts), skills. Nothing in it changes between calls, so the whole thing can be cached.

In [5]:
RULES = """
You are an atmospheric scientist analysing AERONET Level 2.0 aerosol optical depth (AOD) data with Python.
The data is a single parquet table in a sandbox you reach through the run_python tool. Work like an analyst:
- Look at the data before answering. Start with list_sites (exact name, coordinates, record span) and
  describe_columns(site) (which wavelengths that site measured); write code for the analysis, not for that.
  Never guess numbers.
- Follow the matching skill below step by step; say which skill and which thresholds you used.
- Small steps: one run_python call per step, check the output, then continue. Variables persist between calls.
- Every answer states: site(s), coordinates, period used, wavelength, N (days/months/years), method, and caveats.
  AOD to 3 decimals, Ångström exponent to 2. "No data" is not zero.
- Figures: axis labels with units, title with site and period, legend; saved automatically — give the file name.
  Save tables you produce as CSV under /workspace and give the file name.
- Do not attribute causes (specific fires, policies, storms) that the data cannot show.
- If the question cannot be answered from this table (no such site, period outside the record, needs other data),
  say so plainly instead of approximating.
- AERONET data policy: results using a site's data should acknowledge the site PI (columns PI, PI_Email).
"""

DATA_DICTIONARY = """
### data dictionary — /data/aeronet.parquet (AERONET Version 3, Level 2.0, daily averages, all sites)
One row per site and day. Column names are AERONET's with ( ) [ ] - replaced by _.
- AERONET_Site_Name (str), Site_Latitude_Degrees, Site_Longitude_Degrees (decimal degrees), Site_Elevation_m (m)
- datetime_utc (12:00 UTC), date (datetime), year, month, Day_of_Year
- AOD_<λ>nm: aerosol optical depth at λ nm, unitless; NaN = not measured that day or failed Level 2.0 screening.
  Wavelengths present: 340, 380, 400, 412, 440, 443, 490, 500, 510, 532, 551, 555, 560, 620, 667, 675, 681, 709,
  779, 865, 870, 1020, 1640 (most sites: 340, 380, 440, 500, 675, 870, 1020).
- Angstrom_Exponent_<λ1>_<λ2>nm: Ångström exponent between λ1 and λ2 (340_440, 380_500, 440_675, 440_870,
  500_870; 440_675nm_Polar for polar instruments), unitless.
- Precipitable_Water_cm: column water vapour, cm.
- N_<column>: number of Level 2.0 measurements averaged into that day's value (0 -> value is NaN).
- Data_Quality_Level (always "lev20"), AERONET_Instrument_Number (sun-photometer id; changes = instrument swap),
  PI, PI_Email (site principal investigator).
"""

def environment_block():
    env = sb.start(CONVERSATION_ID)            # (re)starts the container and returns its /env
    return ("### sandbox environment\n"
            f"python {env['python']}; packages: " + ", ".join(f"{k} {v}" for k, v in env["packages"].items()) + "\n"
            f"limits: {env['memory_limit_mb']} MB RAM, {env['cpus']} CPUs, {env['workspace_cap_gb']} GB in /workspace, no network\n"
            f"pre-loaded in the kernel:\n{env['preloaded']}\n")

INSTRUCTIONS = "\n".join([RULES.strip(), DATA_DICTIONARY.strip(), environment_block(), "### skills\n" + SKILLS_TEXT])
print(f"{len(INSTRUCTIONS.split())} words ≈ {len(INSTRUCTIONS) // 4} tokens")

1713 words ≈ 2719 tokens


In [6]:
print(INSTRUCTIONS)

You are an atmospheric scientist analysing AERONET Level 2.0 aerosol optical depth (AOD) data with Python.
The data is a single parquet table in a sandbox you reach through the run_python tool. Work like an analyst:
- Look at the data before answering. Start with list_sites (exact name, coordinates, record span) and
  describe_columns(site) (which wavelengths that site measured); write code for the analysis, not for that.
  Never guess numbers.
- Follow the matching skill below step by step; say which skill and which thresholds you used.
- Small steps: one run_python call per step, check the output, then continue. Variables persist between calls.
- Every answer states: site(s), coordinates, period used, wavelength, N (days/months/years), method, and caveats.
  AOD to 3 decimals, Ångström exponent to 2. "No data" is not zero.
- Figures: axis labels with units, title with site and period, legend; saved automatically — give the file name.
  Save tables you produce as CSV under /workspace 

## 4 · Model + agent

**GPT-5.6 Luna, exactly the way VAYU reaches it:** the OpenAI Responses API on `bedrock-runtime.us-west-2` under
`/openai/v1`, authenticated with a short-lived Bedrock bearer token minted from this box's instance role
(`bedrock:CallWithBearerToken`, 12 h validity — re-run this cell after that). Same endpoint, same model id, same IAM
statements, nothing to change in the console. Pydantic AI's `BedrockMantleResponsesModel` speaks that wire (and fixes
the endpoint's per-response tool-call ids). `REASONING_EFFORT` is the cost/quality knob (`none` … `max`; VAYU runs
`high`). `openai_prompt_cache_key` turns on prompt caching for the static prefix (system prompt + tools).
`retries=2` lets the model fix a badly-formed tool call twice.

In [7]:
from aws_bedrock_token_generator import provide_token
from pydantic_ai import Agent, UsageLimits
from pydantic_ai.models.bedrock_mantle import BedrockMantleResponsesModel, BedrockMantleProvider
from pydantic_ai.models.openai import OpenAIResponsesModelSettings

REGION           = "us-west-2"
BASE_URL         = f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1"   # VAYU's endpoint
MODEL_ID         = "us.openai.gpt-5.6-luna"                                       # VAYU's model
REASONING_EFFORT = "high"                                                         # none | low | medium | high | xhigh | max

model = BedrockMantleResponsesModel(
    MODEL_ID,
    provider=BedrockMantleProvider(base_url=BASE_URL, api_key=provide_token(region=REGION)),
    settings=OpenAIResponsesModelSettings(max_tokens=8000, openai_reasoning_effort=REASONING_EFFORT,
                                          openai_prompt_cache_key="aeronet:v1"),
)

agent = Agent(model, instructions=INSTRUCTIONS, tools=TOOLS, retries=2)
print(model.model_name, "via", model.client.base_url)

us.openai.gpt-5.6-luna via https://bedrock-runtime.us-west-2.amazonaws.com/openai/v1/


## 5 · `ask()` — one conversation, many questions

`history` is the conversation. Each `ask()` sends the question plus the whole history, runs the tool loop until the
model answers in text, then appends this turn's messages (question, tool calls, tool results, answer) to `history`.
`UsageLimits(request_limit=…)` stops a run that loops on errors.

In [8]:
from pydantic_ai.messages import ModelRequest, ToolReturnPart

history = []

def show_figures(messages):
    '''Display PNGs mentioned in this turn's tool results (the model does not see them, you do).'''
    for m in messages:
        if isinstance(m, ModelRequest):
            for part in m.parts:
                if isinstance(part, ToolReturnPart) and isinstance(part.content, str):
                    for line in part.content.splitlines():
                        if line.startswith("figures saved: "):
                            for f in line[len("figures saved: "):].split(", "):
                                display(Image(sb.CONV_DIR / CONVERSATION_ID / f))

def ask(question, max_requests=10):
    result = agent.run_sync(question, message_history=history, usage_limits=UsageLimits(request_limit=max_requests))
    history.extend(result.new_messages())
    print(result.output)
    u = result.usage
    print(f"\n[{u.requests} model calls, {u.tool_calls} tool calls; tokens in {u.input_tokens} / out {u.output_tokens} "
          f"/ cache read {u.cache_read_tokens} / cache write {u.cache_write_tokens}]")
    show_figures(result.new_messages())
    return result

## 6 · Try it

Run one cell at a time. The second question is a follow-up: it only makes sense because `history` carries the first.

In [9]:
r = ask("Which five sites have the longest AOD 500 nm records, and how many days of data does each have?")

## Longest AOD 500 nm records

**Skill used:** `data_access`  
**Threshold:** none. I counted every daily Level 2.0 record with a non-missing `AOD_500nm`; these counts exactly matched days with `N_AOD_500nm > 0`. No monthly or annual completeness threshold was applied.

| Rank | Site | Coordinates (°N, °E) | Elevation (m) | Valid AOD 500 nm period | N valid days | Coverage of site-days |
|---:|---|---:|---:|---|---:|---:|
| 1 | **Mauna_Loa** | 19.536, −155.577 | 3,402 | 1995-03-16 to 2026-07-07 | **9,152** | 99.5% |
| 2 | **GSFC** | 38.993, −76.840 | 87 | 1995-03-24 to 2026-01-08 | **7,973** | 96.8% |
| 3 | **SEDE_BOKER** | 30.855, 34.782 | 480 | 1997-09-25 to 2025-04-04 | **7,706** | 99.2% |
| 4 | **Izana** | 28.309, −16.499 | 2,401 | 1997-06-15 to 2026-01-27 | **6,375** | 98.2% |
| 5 | **Sevilleta** | 34.355, −106.885 | 1,477 | 1995-07-06 to 2025-03-26 | **6,204** | 96.1% |

The counts are **daily means**, with one row per site-day. Missing AOD values were treated as missing data, no

In [18]:
r.all_messages()

[ModelRequest(parts=[UserPromptPart(content='Which five sites have the longest AOD 500 nm records, and how many days of data does each have?', timestamp=datetime.datetime(2026, 9, 26, 2, 40, 38, 354210, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 9, 26, 2, 40, 38, 355028, tzinfo=datetime.timezone.utc), instructions='You are an atmospheric scientist analysing AERONET Level 2.0 aerosol optical depth (AOD) data with Python.\nThe data is a single parquet table in a sandbox you reach through the run_python tool. Work like an analyst:\n- Look at the data before answering. Start with list_sites (exact name, coordinates, record span) and\n  describe_columns(site) (which wavelengths that site measured); write code for the analysis, not for that.\n  Never guess numbers.\n- Follow the matching skill below step by step; say which skill and which thresholds you used.\n- Small steps: one run_python call per step, check the output, then continue. Variables persist between calls

In [41]:
print(r.all_messages()[0])

ModelRequest(parts=[UserPromptPart(content='Which five sites have the longest AOD 500 nm records, and how many days of data does each have?', timestamp=datetime.datetime(2026, 9, 26, 2, 40, 38, 354210, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 9, 26, 2, 40, 38, 355028, tzinfo=datetime.timezone.utc), instructions='You are an atmospheric scientist analysing AERONET Level 2.0 aerosol optical depth (AOD) data with Python.\nThe data is a single parquet table in a sandbox you reach through the run_python tool. Work like an analyst:\n- Look at the data before answering. Start with list_sites (exact name, coordinates, record span) and\n  describe_columns(site) (which wavelengths that site measured); write code for the analysis, not for that.\n  Never guess numbers.\n- Follow the matching skill below step by step; say which skill and which thresholds you used.\n- Small steps: one run_python call per step, check the output, then continue. Variables persist between calls.

In [42]:
r.all_messages()[0].parts

[UserPromptPart(content='Which five sites have the longest AOD 500 nm records, and how many days of data does each have?', timestamp=datetime.datetime(2026, 9, 26, 2, 40, 38, 354210, tzinfo=datetime.timezone.utc))]

In [43]:
r.all_messages()[0].parts[0].content

'Which five sites have the longest AOD 500 nm records, and how many days of data does each have?'

In [50]:
r.all_messages()[1]

ModelResponse(parts=[ThinkingPart(content='', id='rs_c4c82aa9c3ca56878d74fbf24bae8ee9', signature='rsn_5ZVrif4J0bXIqmWdledj7QIFeNkslIwP99NFFMw/MeNBHXZuNgoApsHQAOIM/tPS5wSRAF8AAQAVYXdzLWNyeXB0by1wdWJsaWMta2V5AERBLzhVOFRjaDA1TXFTY2tIUDIrTU1pSnR6YklveVR2QjBkZUhHQ242NzFuTDBRT2EvTThXZTJVTFI0djc2Z0ZYTlE9PQABABFhd3Mta21zLWhpZXJhcmNoeQAKYmVybS1zdG9yZQBcxl8e+JhzrKO7MHJ3qFfQDs+hbq1CZph0uBPQOVrKc+5ls0wRpnX+0wjWt1KiVKUUU49GThTN6xWTWtZam/iTWO/Gs1t6y5wv1eidXnxCNG5jnpGfb2yzJnrh47YCAAAQAC4IT1Xp+88a1Cad/ic8lLRArOUKa7/MxHHXNc4DTlibPWeIURjytJLexHVC++4LAv////8AAAABAAAAAAAAAAAAAAABAAACVXytIEyV9PY9JYXMwravq+yVCMoitFF7b1axL6nRbdARtUX/V3lrnh/MuIUaZUhhEhZ2LxvXhkIfT0a7KiTqDCKSLSjQuYxTHgr9iSfY6UuLgF+oJEm2GOoCVWosN/meUVls6lN3Lj7XGhRYa13S0DY8VquDVarG+M7c6wFu0OtEP2wIFRIbNnu1wBhFJ4ECL/Ot+AH6wvCld2/bkariOFouEFRSVqhkjs2+TmBz9PFJraONyYTdhfXjaJXD/x391P7hm1OcYsCt1F8RwKUsoHzXwwCgcbCcyNv0GZfOC0+YEHi5PDq0atFPyEDx3y8zaMQcRoPJBhCfx9GTEzr1h01ptFJXnSY9RQsmt0k8znM5MpqlT9CPe1Do5P9cvhtjvsB3HKnwWTwuQdkPO/Tw39TdmzmKmtNxqKaCQTq8/GNs9

In [28]:
r.all_messages()[0].parts[0].content

'Which five sites have the longest AOD 500 nm records, and how many days of data does each have?'

In [19]:
r.usage

RunUsage(cache_read_tokens=49755, cache_write_tokens=11935, cost=Decimal('0.011064735'), details={'reasoning_tokens': 2948}, input_tokens=61706, output_reasoning_tokens=2948, output_tokens=5064, requests=8, tool_calls=7)

In [12]:
print(r.output)

## Longest AOD 500 nm records

**Skill used:** `data_access`  
**Threshold:** none. I counted every daily Level 2.0 record with a non-missing `AOD_500nm`; these counts exactly matched days with `N_AOD_500nm > 0`. No monthly or annual completeness threshold was applied.

| Rank | Site | Coordinates (°N, °E) | Elevation (m) | Valid AOD 500 nm period | N valid days | Coverage of site-days |
|---:|---|---:|---:|---|---:|---:|
| 1 | **Mauna_Loa** | 19.536, −155.577 | 3,402 | 1995-03-16 to 2026-07-07 | **9,152** | 99.5% |
| 2 | **GSFC** | 38.993, −76.840 | 87 | 1995-03-24 to 2026-01-08 | **7,973** | 96.8% |
| 3 | **SEDE_BOKER** | 30.855, 34.782 | 480 | 1997-09-25 to 2025-04-04 | **7,706** | 99.2% |
| 4 | **Izana** | 28.309, −16.499 | 2,401 | 1997-06-15 to 2026-01-27 | **6,375** | 98.2% |
| 5 | **Sevilleta** | 34.355, −106.885 | 1,477 | 1995-07-06 to 2025-03-26 | **6,204** | 96.1% |

The counts are **daily means**, with one row per site-day. Missing AOD values were treated as missing data, no

In [ ]:
r = ask("Show the monthly climatology of AOD 500 nm and Ångström exponent 440–870 nm at Banizoumbou.")

In [ ]:
r = ask("Now put Ilorin on the same plots and tell me how the two seasonal cycles differ.")

In [ ]:
r = ask("Is there a long-term trend in AOD 500 nm at GSFC?")

In [ ]:
r = ask("What kind of aerosol dominates at Kanpur in spring versus autumn?")

## 7 · What the model actually saw

Every message of a turn, clipped. This is the transcript the paper's traces will store; reading it is how you find
out whether a skill was followed.

In [ ]:
from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart, UserPromptPart

def show_turn(messages, clip=600):
    for m in messages:
        for p in m.parts:
            if isinstance(p, UserPromptPart):   print(f"\n>>> USER: {p.content}")
            elif isinstance(p, ToolCallPart):   print(f"\n--- TOOL CALL {p.tool_name}:\n{str(p.args)[:clip]}")
            elif isinstance(p, ToolReturnPart): print(f"\n--- TOOL RESULT {p.tool_name}:\n{str(p.content)[:clip]}")
            elif isinstance(p, TextPart):       print(f"\n<<< MODEL: {p.content[:clip]}")
        if isinstance(m, ModelResponse) and m.usage:
            print(f"    [in {m.usage.input_tokens} / out {m.usage.output_tokens} / cache read {m.usage.cache_read_tokens}]")

show_turn(r.new_messages())

## 8 · Save / resume a conversation

`history` is plain data, so it can be written to JSON in the conversation folder and loaded tomorrow. What does NOT
survive: the sandbox variables (the container stops itself after 60 idle minutes). Files in `conversation_data/<id>/`
do, so the model can reload a saved CSV/parquet on the next question — tell it so in the first question after a
resume, e.g. "the kernel was restarted; earlier results are in /workspace".

In [ ]:
from pydantic_ai.messages import ModelMessagesTypeAdapter

MESSAGES_FILE = sb.CONV_DIR / CONVERSATION_ID / "messages.json"

def save_history():
    MESSAGES_FILE.write_bytes(ModelMessagesTypeAdapter.dump_json(history, indent=1))
    print(f"{len(history)} messages -> {MESSAGES_FILE}")

def load_history():
    global history
    history = ModelMessagesTypeAdapter.validate_json(MESSAGES_FILE.read_bytes())
    print(f"{len(history)} messages <- {MESSAGES_FILE}")

save_history()

In [ ]:
# tomorrow: run the setup cells (sections 1–5), then
# load_history()
# r = ask("The kernel was restarted; earlier results are in /workspace. Where did we stop, and what do you still have on disk?")

## 9 · Cleanup

In [ ]:
sb.stop(CONVERSATION_ID)